# 07 Bulk TCRseq. Annotation and analysis

In [ ]:
import anndata as ad
import scanpy as sc
import scanpy.external as sce
import scirpy as ir
import pandas as pd
import numpy as np
import seaborn as sb
import os
import muon as mu
import scrublet as scr
import math
from scipy.stats import binomtest
from scipy.stats import mannwhitneyu
from scipy.stats import binomtest
from scipy.stats import spearmanr
from scipy.stats import wilcoxon
from scipy.stats import multinomial
from scipy.stats import fisher_exact
from scipy.stats import sem

# Helps to define plotting area
import matplotlib.pyplot as plt
from matplotlib import rcParams 
from matplotlib.pyplot import rc_context
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib import colors
from matplotlib_venn import venn2
import matplotlib.patches as mpatches

# 
from sklearn.linear_model import LinearRegression

## For making text editable in .pdfs once exported use cairo backend instead of agg
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rc('font', size=14)
plt.rcParams['pdf.fonttype'] = 42

#Set global figure size
# plt.rcParams["figure.figsize"] = (3.5,3)

# set seed
np.random.seed(44)

#display options
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999

In [ ]:
# Current working folder
os.getcwd()

### Helper functions from Mimi

In [ ]:
tcr_gene_prefixs = ['TRAV', 'TRAJ', 'TRAC', 'TRB', 'TRDV', 'TRDC', 'TRG']

def filter_tcr_genes(adata):
    non_tcr_genes = adata.var_names
    for prefix in tcr_gene_prefixs:
        non_tcr_genes = [el for el in non_tcr_genes if not el.startswith(prefix)]
    adata = adata[:, non_tcr_genes]
    return adata

def filter_high_var(adata, n):
    #sc.pp.highly_variable_genes(
    #adata,
    #flavor="seurat_v3",
    #n_top_genes=n,
    #layer="counts",
    #batch_key="pool",
    #subset=True,
    #span = 0.75) #default = 0.3. It's the fraction of cells used for estimation of hvg - if I have 150 cells in one sample but 150.000 cells in total then this sample becomes an issue. 
    
    sc.pp.highly_variable_genes(adata, n_top_genes=n)
    adata = adata[:, adata.var['highly_variable']].copy()
    return adata


def calculate_umap(adata, n_high_var=5000, remove_tcr_genes=True):
    """
    Calculates UMAP coordinates
    :param adata: AnnData object
    :param n_high_var: how many variable genes to use, None uses raw data
    :param remove_tcr_genes: whether to remove TCR genes
    :return: results stored in adata object
    """
    adata_tmp = adata.copy()

    if remove_tcr_genes:
        adata_tmp = filter_tcr_genes(adata_tmp)
    if n_high_var is not None:
        adata_tmp = filter_high_var(adata_tmp, n_high_var)

    sc.pp.neighbors(adata_tmp, n_pcs=20)            
    sc.tl.umap(adata_tmp)
    adata.obsm['X_umap'] = adata_tmp.obsm['X_umap']


def calculate_leiden(adata, resolution, n_high_var=5000, remove_tcr_genes=True):
    adata_tmp = adata.copy()

    if remove_tcr_genes:
        adata_tmp = filter_tcr_genes(adata_tmp)
    if n_high_var is not None:
        adata_tmp = filter_high_var(adata_tmp, n_high_var)

    sc.pp.neighbors(adata_tmp, n_pcs=20)
    sc.tl.leiden(adata_tmp, resolution=resolution)

    leiden_column_name = f"leiden_{resolution}"
    adata.obs[leiden_column_name] = adata_tmp.obs['leiden']

# Loading data
See the following for a discussion on data concatenation in muon: https://github.com/scverse/muon/issues/127

In [ ]:
os.getcwd()

In [ ]:
path_base = "..\\..\\04_data_objects\\01_preprocessed_scseq_scanpy"

In [ ]:
sdata = mu.read(filename="..\\..\\04_data_objects\\01_preprocessed_scseq_scanpy\\2025_4 - collected datasets - including doublets\\2025_CollectedScData_w_L2_ann_fin.h5mu")

In [ ]:
sdata

### Number of cells 

In [ ]:
sdata["gex"].obs.groupby("Compartment").size()

In [ ]:
sdata["gex"].obs.columns

### Export for R    

In [ ]:
sdata["gex"].obs.columns

In [ ]:
# Define clone_handle - a combination of the donor information and the clone_id information to make each shared clone unique in each donor. 
sdata["gex"].obs["IR_VDJ_1_junction_aa"] = sdata["airr"].obs["IR_VDJ_1_junction_aa"].copy()
sdata["gex"].obs["IR_VDJ_2_junction_aa"] = sdata["airr"].obs["IR_VDJ_2_junction_aa"].copy()
sdata["gex"].obs["IR_VJ_1_junction_aa"] = sdata["airr"].obs["IR_VJ_1_junction_aa"].copy()
sdata["gex"].obs["IR_VJ_2_junction_aa"] = sdata["airr"].obs["IR_VJ_2_junction_aa"].copy()
sdata["gex"].obs["donor_VDJ"] = sdata["gex"].obs["donor"].astype(str) + "_" + sdata["gex"].obs["IR_VDJ_1_junction_aa"].astype(str)

In [ ]:
sdata["gex"].obs[sdata["gex"].obs["clone_handle"] == "Pt1-1462"][["L1_ann", "IR_VDJ_1_junction_aa", "donor_VDJ", "IR_VJ_1_junction_aa", "chain_pairing"]].head(5)

In [ ]:
sdata["gex"].obs[sdata["gex"].obs["clone_handle"] == "Pt1-225"][["L1_ann", "IR_VDJ_1_junction_aa", "donor_VDJ", "IR_VJ_1_junction_aa", "chain_pairing"]].head(5)

In [ ]:
sdata["gex"].obs[["donor_VDJ", "Compartment", "L2_ann_fin_vote", "clone_handle", "L1_ann", "chain_pairing"]].to_csv("..\\..\\04_data_objects\\04_bulk_TCRseq_ouputs\\20251030 - e78-e80 - LCL bulkTCRseq Pt14 to Pt21\\20251030_scseq_VDJ_handles.csv")

## Load Bulk TCRseq data - e78-e80

In [ ]:
TCRseq1 = pd.read_csv("..\\..\\04_data_objects\\04_bulk_TCRseq_ouputs\\20251030 - e78-e80 - LCL bulkTCRseq Pt14 to Pt21\\20251030_AllTimepoints_UniqueTRBs.csv")
TCRseq1.head(2)

In [ ]:
sdata["gex"].obs["TRB_handle"] = sdata["gex"].obs["donor"].astype(str) + "_" + sdata["gex"].obs["IR_VDJ_1_junction_aa"].astype(str)

In [ ]:
sdata["gex"].obs["TRB_handle"]

In [ ]:
# Assume df contains the columns: "donor_VDJ" and feature columns to be added

# Create a mapping from donor_VDJ to each feature
df_features = TCRseq1.set_index("TRB_handle")

# Align based on donor_VDJ, preserving the order of sdata["gex"].obs
obs = sdata["gex"].obs.copy()

# For each feature column, map the corresponding values
for col in df_features.columns:
    obs[col] = obs["TRB_handle"].map(df_features[col])

# Reassign the modified DataFrame to the AnnData object
sdata["gex"].obs = obs

In [ ]:
# Replace 0 with NaN in the relevant columns of obs
cols = ["D0", "D19", "D37"]
sdata["gex"].obs[cols] = sdata["gex"].obs[cols].replace(0, np.nan)

In [ ]:
p1 = sc.pl.umap(
    sdata["gex"], 
    color=["D0", "D19", "D37"], 
    size=10, 
    add_outline=True,
    legend_fontoutline=2, 
    legend_fontweight='bold', 
    legend_fontsize="6",  
    cmap="RdYlBu_r",  # diverging colormap to emphasize positive/negative deviations
    frameon=False, 
    show=False,
    return_fig=True
)

p1.savefig("..\\..\\05_plots\\08_AfterAntwerp\\71_LCL_fraUMI_umap.pdf", bbox_inches='tight', format="pdf", dpi = 200)

In [ ]:
p1 = sc.pl.umap(
    sdata["gex"], 
    color=["Compartment", "L2_ann_fin_vote", "CD8A", "CD4", 
           "GZMB", "PRF1", "GZMK","CXCR3"], 
    size=2, 
    add_outline=True,
    legend_fontoutline=2, 
    legend_fontweight='bold', 
    legend_fontsize="6",  
    cmap="RdYlBu_r",  # diverging colormap to emphasize positive/negative deviations
    frameon=False, 
    show=False,
    return_fig=True
)

p1.savefig("..\\..\\05_plots\\08_AfterAntwerp\\71_LCL_orientation_umap.pdf", bbox_inches='tight', format="pdf", dpi = 200)